
# Centered Gaussian covariance flow, sampled by covariance arclength

This notebook illustrates the covariance evolution of the spectral Wasserstein
gradient flow of $\mathrm{KL}(\mu\,|\,\nu)$, restricted to centered Gaussian
measures.

Both the initial Gaussian and the target Gaussian have zero mean, so only the
covariance matrix evolves. In the figures, the covariance ellipses are
artificially translated horizontally so that the evolution can be read from
left to right without overlap. This horizontal translation is only a display
device and is not part of the dynamics.

To make the deformation visible, snapshots are not sampled uniformly in time.
Instead, they are sampled uniformly with respect to arclength in covariance
space, using the coordinates $(\Sigma_{11},\Sigma_{12},\Sigma_{22})$.
We also produce an early-arclength zoom showing only the first part of the
covariance trajectory.



## Centered Gaussian ODE

Let $\mu_t=\mathcal N(0,\Sigma_t)$ and
$\nu=\mathcal N(0,\bar\Sigma)$. The Wasserstein gradient of
$\mathrm{KL}(\mu_t\,|\,\nu)$ is linear:

$$
g_t(x)=H_t x,
\qquad
H_t=\bar\Sigma^{-1}-\Sigma_t^{-1}.
$$

The Fisher-type matrix entering the Schatten geometry is

$$
F_t=H_t\Sigma_tH_t.
$$

For Schatten parameter $p$, set $q=2p/(2p-1)$ and

$$
P_{p,t}=\mathcal N_t^{2-q}F_t^{q/2-1},
\qquad
\mathcal N_t^q=\mathrm{tr}(F_t^{q/2}).
$$

The covariance ODE is

$$
\dot\Sigma_t=-P_{p,t}H_t\Sigma_t-\Sigma_tH_tP_{p,t}.
$$

For $p=1$, $P_{1,t}=I$. For $p=\infty$,
$P_{\infty,t}=\mathrm{tr}(F_t^{1/2})F_t^{-1/2}$.


## Helpers


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from pathlib import Path

outdir = Path(".")

def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def cov_from_axes(eigs, theta):
    R = rot(theta)
    return R @ np.diag(eigs) @ R.T

def sym(A):
    return 0.5 * (A + A.T)

def sqrt_spd(A, eps=1e-12):
    vals, vecs = np.linalg.eigh(sym(A))
    vals = np.maximum(vals, eps)
    return vecs @ np.diag(np.sqrt(vals)) @ vecs.T

def invsqrt_spd(A, eps=1e-10):
    vals, vecs = np.linalg.eigh(sym(A))
    vals = np.maximum(vals, eps)
    return vecs @ np.diag(vals**-0.5) @ vecs.T

def pack_cov(S):
    return np.array([S[0, 0], S[0, 1], S[1, 1]], dtype=float)

def unpack_cov(y):
    return sym(np.array([[y[0], y[1]], [y[1], y[2]]], dtype=float))

def ellipse_patch(center, Sigma, nsig=1.7):
    vals, vecs = np.linalg.eigh(sym(Sigma))
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    return Ellipse(center, 2*nsig*np.sqrt(vals[0]), 2*nsig*np.sqrt(vals[1]), angle=angle)

## Centered covariance setup

The two means are zero. The initial and target covariances have the same eigenvalues but different orientations.

In [ ]:
Sigma_initial = cov_from_axes([1.0, 0.1], np.deg2rad(-35.0))
Sigma_target = cov_from_axes([1.0, 0.1], np.deg2rad(35.0))

print("Initial covariance:\n", Sigma_initial)
print("Target covariance:\n", Sigma_target)

## ODE integration

For $p=\infty$, `reparametrize=True` changes only the scalar speed along the same covariance path.

In [ ]:
def covariance_rhs(Sigma, p):
    inv_target = np.linalg.inv(Sigma_target)
    inv_Sigma = np.linalg.inv(Sigma)
    H = inv_target - inv_Sigma
    F = sym(H @ Sigma @ H)

    if p == 1:
        P = np.eye(2)
    elif np.isinf(p):
        P = np.trace(sqrt_spd(F)) * invsqrt_spd(F)
    else:
        q = 2 * p / (2 * p - 1)
        vals, vecs = np.linalg.eigh(F)
        vals = np.maximum(vals, 1e-12)
        N = np.sum(vals ** (q / 2)) ** (1 / q)
        P = (N ** (2 - q)) * (vecs @ np.diag(vals ** (q / 2 - 1)) @ vecs.T)

    dSigma = -P @ H @ Sigma - Sigma @ H @ P
    return sym(dSigma)

def integrate_covariance_flow(p, T=8.0, dt=0.001, reparametrize=False):
    n_steps = int(round(T / dt))
    y = pack_cov(Sigma_initial)
    times = np.linspace(0.0, T, n_steps + 1)
    Y = np.empty((n_steps + 1, 3))
    Y[0] = y

    def rhs_y(y):
        S = unpack_cov(y)
        dS = covariance_rhs(S, p)
        dy = np.array([dS[0, 0], dS[0, 1], dS[1, 1]], dtype=float)

        if reparametrize:
            # Positive scalar time change: same covariance path, different speed.
            speed = np.linalg.norm(dy)
            if speed > 1e-12:
                dy = dy / speed
        return dy

    for k in range(n_steps):
        k1 = rhs_y(y)
        k2 = rhs_y(y + 0.5 * dt * k1)
        k3 = rhs_y(y + 0.5 * dt * k2)
        k4 = rhs_y(y + dt * k3)
        y = y + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

        # Gentle numerical projection to SPD covariances.
        S = unpack_cov(y)
        vals, vecs = np.linalg.eigh(S)
        vals = np.maximum(vals, 1e-8)
        S = vecs @ np.diag(vals) @ vecs.T
        y = pack_cov(S)
        Y[k+1] = y

    return times, Y

## Arclength sampling and display

Snapshots are selected uniformly in covariance arclength, not uniformly in time. This makes the visible deformation more balanced.

In [ ]:
def covariance_arclength_indices(Y, n=22, fraction=(0.0, 1.0)):
    """Sample indices uniformly in covariance-space arclength."""
    dY = np.diff(Y, axis=0)
    increments = np.linalg.norm(dY, axis=1)
    arclen = np.concatenate([[0.0], np.cumsum(increments)])
    total = arclen[-1]
    if total <= 1e-14:
        return np.linspace(0, len(Y)-1, n, dtype=int)

    a, b = fraction
    targets = np.linspace(a * total, b * total, n)
    idx = np.searchsorted(arclen, targets)
    idx = np.clip(idx, 0, len(Y)-1)

    idx_unique = []
    for i in idx:
        i = int(i)
        if len(idx_unique) == 0 or i != idx_unique[-1]:
            idx_unique.append(i)
    return np.array(idx_unique, dtype=int)

def plot_covariance_gallery(Y, pdf_path, n_ellipses=20, nsig=1.7,
                            spacing=3.5, fraction=(0.0, 1.0)):
    idx = covariance_arclength_indices(Y, n=n_ellipses, fraction=fraction)
    cmap = plt.get_cmap("coolwarm")
    fig, ax = plt.subplots(figsize=(9.0, 2.2))

    for j, k in enumerate(idx):
        S = unpack_cov(Y[k])
        center = np.array([j * spacing, 0.0])  # display-only translation
        color = cmap(j / max(len(idx)-1, 1))

        e = ellipse_patch(center, S, nsig=nsig)
        e.set_facecolor(color)
        e.set_edgecolor(color)
        e.set_alpha(0.75)
        e.set_linewidth(0.9)
        ax.add_patch(e)

    # Target shown as a red outline at the last display position.
    target_center = np.array([(len(idx)-1) * spacing, 0.0])
    target = ellipse_patch(target_center, Sigma_target, nsig=nsig)
    target.set_facecolor("none")
    target.set_edgecolor("red")
    target.set_linewidth(2.2)
    ax.add_patch(target)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-2.0, (len(idx)-1)*spacing + 2.0)
    ax.set_ylim(-2.0, 2.0)
    ax.set_axis_off()

    fig.tight_layout(pad=0)
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.02)
    plt.show()

## Full covariance trajectory


In [ ]:
times_p1, Y_p1 = integrate_covariance_flow(1, T=8.0, dt=0.001)
times_pinf, Y_pinf = integrate_covariance_flow(np.inf, T=18.0, dt=0.002, reparametrize=True)

plot_covariance_gallery(
    Y_p1,
    "gaussian_covariance_flow_centered_p1_arclength.pdf",
    n_ellipses=20,
    fraction=(0.0, 1.0),
)

plot_covariance_gallery(
    Y_pinf,
    "gaussian_covariance_flow_centered_pinf_arclength.pdf",
    n_ellipses=20,
    fraction=(0.0, 1.0),
)

## Early covariance-arclength zoom

This zoom uses only the first 25% of the covariance-space arclength, with larger ellipses and spacing.

In [ ]:
# Early zoom: show only the first 25% of the covariance-space arclength.
plot_covariance_gallery(
    Y_p1,
    "gaussian_covariance_flow_centered_p1_arclength_early_zoom.pdf",
    n_ellipses=18,
    nsig=2.2,
    spacing=4.2,
    fraction=(0.0, 0.25),
)

plot_covariance_gallery(
    Y_pinf,
    "gaussian_covariance_flow_centered_pinf_arclength_early_zoom.pdf",
    n_ellipses=18,
    nsig=2.2,
    spacing=4.2,
    fraction=(0.0, 0.25),
)